<a href="https://colab.research.google.com/github/heyitsarun/Aero-Gen-Synthetic-Anomaly-Generation-in-Aviation-Communications/blob/main/03_detection_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

GPU available: True
Device name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/aerogen_project'
DATA_DIR = f'{PROJECT_ROOT}/data'
GEN_DIR = f'{PROJECT_ROOT}/generated'
EVAL_DIR = f'{PROJECT_ROOT}/evaluation'
os.makedirs(EVAL_DIR, exist_ok=True)

# Real, normal ATC transcripts (negative class)
real_df = pd.read_csv(f'{DATA_DIR}/atcosim_enriched.csv')
real_df = real_df[['text']].copy()
real_df['label'] = 0
real_df['generation_method'] = 'real'
real_df['anomaly_type'] = 'none'

# Rule-based synthetic anomalies (positive class)
rule_df = pd.read_csv(f'{GEN_DIR}/synthetic_rule_based.csv')
rule_df = rule_df[['corrupted_text', 'anomaly_type']].copy()
rule_df.columns = ['text', 'anomaly_type']
rule_df['label'] = 1
rule_df['generation_method'] = 'rule_based'

# LLM-generated synthetic anomalies (positive class)
llm_df = pd.read_csv(f'{GEN_DIR}/synthetic_llm_generated.csv')
llm_df = llm_df[['corrupted_text', 'anomaly_type']].copy()
llm_df.columns = ['text', 'anomaly_type']
llm_df['label'] = 1
llm_df['generation_method'] = 'llm_few_shot'

print(f'Real (normal): {len(real_df)}')
print(f'Rule-based synthetic (anomalous): {len(rule_df)}')
print(f'LLM-generated synthetic (anomalous): {len(llm_df)}')

combined_df = pd.concat([real_df, rule_df, llm_df], ignore_index=True)
combined_df = combined_df.dropna(subset=['text']).reset_index(drop=True)
combined_df['text'] = combined_df['text'].astype(str).str.strip()
combined_df = combined_df[combined_df['text'].str.len() > 0].reset_index(drop=True)

print(f'\nTotal combined rows: {len(combined_df)}')
print(combined_df['label'].value_counts())
print(combined_df.groupby(['label','generation_method']).size())

Mounted at /content/drive
Real (normal): 6519
Rule-based synthetic (anomalous): 1312
LLM-generated synthetic (anomalous): 480

Total combined rows: 8311
label
0    6519
1    1792
Name: count, dtype: int64
label  generation_method
0      real                 6519
1      llm_few_shot          480
       rule_based           1312
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

# Split each source independently (stratified by anomaly_type where relevant),
# so we can later assemble different TRAIN combinations while sharing one fixed TEST set.
real_train, real_test = train_test_split(real_df, test_size=0.2, random_state=42)
rule_train, rule_test = train_test_split(rule_df, test_size=0.2, random_state=42, stratify=rule_df['anomaly_type'])
llm_train, llm_test = train_test_split(llm_df, test_size=0.2, random_state=42, stratify=llm_df['anomaly_type'])

print(f'Real: {len(real_train)} train / {len(real_test)} test')
print(f'Rule-based: {len(rule_train)} train / {len(rule_test)} test')
print(f'LLM: {len(llm_train)} train / {len(llm_test)} test')

# Fixed test set used for ALL model evaluations (fair comparison across configurations)
test_df = pd.concat([real_test, rule_test, llm_test], ignore_index=True)
print(f'\nFixed test set: {len(test_df)} rows, label distribution:')
print(test_df['label'].value_counts())

for path, d in [('real_train', real_train), ('real_test', real_test),
                ('rule_train', rule_train), ('rule_test', rule_test),
                ('llm_train', llm_train), ('llm_test', llm_test),
                ('test_combined', test_df)]:
    d.to_csv(f'{EVAL_DIR}/{path}.csv', index=False)
print('\nSaved all splits to', EVAL_DIR)

Real: 5215 train / 1304 test
Rule-based: 1049 train / 263 test
LLM: 384 train / 96 test

Fixed test set: 1663 rows, label distribution:
label
0    1304
1     359
Name: count, dtype: int64

Saved all splits to /content/drive/MyDrive/aerogen_project/evaluation


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

def build_training_set(*dfs):
    return pd.concat(dfs, ignore_index=True)

def evaluate_baseline(train_df, test_df, label):
    n_classes = train_df['label'].nunique()

    if n_classes < 2:
        # No anomaly examples to learn from — simulate the trivial "always predict normal" baseline
        preds = np.zeros(len(test_df), dtype=int)
        f1 = f1_score(test_df['label'], preds)
        print(f'=== {label} ===')
        print('(Only one class present in training — trivial always-predict-normal baseline)')
        print(classification_report(test_df['label'], preds, target_names=['normal','anomalous'], zero_division=0))
        return f1

    vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
    X_train = vectorizer.fit_transform(train_df['text'])
    X_test = vectorizer.transform(test_df['text'])

    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train, train_df['label'])

    preds = clf.predict(X_test)
    f1 = f1_score(test_df['label'], preds)
    print(f'=== {label} ===')
    print(classification_report(test_df['label'], preds, target_names=['normal','anomalous'], zero_division=0))
    return f1

configs = {
    'Real-only (no anomalies in training)': build_training_set(real_train),
    'Real + Rule-based': build_training_set(real_train, rule_train),
    'Real + LLM-generated': build_training_set(real_train, llm_train),
    'Real + Both': build_training_set(real_train, rule_train, llm_train),
}

baseline_results = {}
for name, tdf in configs.items():
    baseline_results[name] = evaluate_baseline(tdf, test_df, name)
    print()

print('=== F1 Summary ===')
for name, f1 in baseline_results.items():
    print(f'{name}: {f1:.3f}')

=== Real-only (no anomalies in training) ===
(Only one class present in training — trivial always-predict-normal baseline)
              precision    recall  f1-score   support

      normal       0.78      1.00      0.88      1304
   anomalous       0.00      0.00      0.00       359

    accuracy                           0.78      1663
   macro avg       0.39      0.50      0.44      1663
weighted avg       0.61      0.78      0.69      1663


=== Real + Rule-based ===
              precision    recall  f1-score   support

      normal       0.86      0.77      0.81      1304
   anomalous       0.39      0.54      0.46       359

    accuracy                           0.72      1663
   macro avg       0.63      0.66      0.63      1663
weighted avg       0.76      0.72      0.74      1663


=== Real + LLM-generated ===
              precision    recall  f1-score   support

      normal       0.82      0.85      0.84      1304
   anomalous       0.37      0.31      0.34       359

  

In [5]:
baseline_summary = pd.DataFrame([
    {'config': name, 'f1_anomalous': f1, 'model': 'tfidf_logreg'}
    for name, f1 in baseline_results.items()
])
baseline_summary.to_csv(f'{EVAL_DIR}/baseline_results.csv', index=False)
print(baseline_summary)

                                 config  f1_anomalous         model
0  Real-only (no anomalies in training)      0.000000  tfidf_logreg
1                     Real + Rule-based      0.456674  tfidf_logreg
2                  Real + LLM-generated      0.335855  tfidf_logreg
3                           Real + Both      0.456818  tfidf_logreg


In [6]:
!pip install -q transformers datasets accelerate

import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
EVAL_DIR = f'{PROJECT_ROOT}/evaluation'

real_train = pd.read_csv(f'{EVAL_DIR}/real_train.csv')
rule_train = pd.read_csv(f'{EVAL_DIR}/rule_train.csv')
llm_train = pd.read_csv(f'{EVAL_DIR}/llm_train.csv')
test_df = pd.read_csv(f'{EVAL_DIR}/test_combined.csv')

print(f'real_train: {len(real_train)}, rule_train: {len(rule_train)}, llm_train: {len(llm_train)}, test: {len(test_df)}')

real_train: 5215, rule_train: 1049, llm_train: 384, test: 1663


In [8]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def train_and_evaluate(train_df, test_df, config_name, output_dir):
    train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
    test_ds = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

    train_ds = train_ds.map(tokenize_function, batched=True)
    test_ds = test_ds.map(tokenize_function, batched=True)

    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=50,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    print(f'\n{"="*20} Training: {config_name} {"="*20}')
    trainer.train()
    results = trainer.evaluate()
    print(f'{config_name} final results:', results)
    return results

In [9]:
test_config_df = pd.concat([real_train, llm_train], ignore_index=True)
test_result = train_and_evaluate(test_config_df, test_df, 'TEST RUN: Real + LLM', f'{EVAL_DIR}/test_run')

Map:   0%|          | 0/5599 [00:00<?, ? examples/s]

Map:   0%|          | 0/1663 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



==================== Training: TEST RUN: Real + LLM ====================


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.232523,0.615952,0.323462,0.887500,0.197772
2,0.199683,0.547373,0.301887,0.984615,0.178273
3,0.160084,0.643179,0.317241,0.907895,0.192201


Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.160084,0.643179,3,0.317241,0.907895,0.192201


TEST RUN: Real + LLM final results: {'eval_loss': 0.6431785821914673, 'eval_f1': 0.31724137931034485, 'eval_precision': 0.9078947368421053, 'eval_recall': 0.19220055710306408}


In [12]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def train_and_evaluate(train_df, test_df, config_name, output_dir):
    train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
    test_ds = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))
    train_ds = train_ds.map(tokenize_function, batched=True)
    test_ds = test_ds.map(tokenize_function, batched=True)

    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    # Class weights inversely proportional to frequency — same principle as class_weight='balanced'
    n_normal = (train_df['label'] == 0).sum()
    n_anomalous = (train_df['label'] == 1).sum()
    total = n_normal + n_anomalous
    class_weights = torch.tensor([total / (2 * n_normal), total / (2 * n_anomalous)], dtype=torch.float)
    print(f'  Class weights (normal, anomalous): {class_weights.tolist()}')

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        logging_steps=50,
        report_to='none',
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    print(f'\n{"="*20} Training: {config_name} {"="*20}')
    trainer.train()
    results = trainer.evaluate()
    print(f'{config_name} final results:', results)
    return results

In [13]:
test_config_df = pd.concat([real_train, llm_train], ignore_index=True)
test_result = train_and_evaluate(test_config_df, test_df, 'TEST RUN: Real + LLM (weighted)', f'{EVAL_DIR}/test_run')

Map:   0%|          | 0/5599 [00:00<?, ? examples/s]

Map:   0%|          | 0/1663 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Class weights (normal, anomalous): [0.5368168950080872, 7.290364742279053]

==================== Training: TEST RUN: Real + LLM (weighted) ====================


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.576328,0.496184,3,0.380454,0.509346,0.303621


TEST RUN: Real + LLM (weighted) final results: {'eval_loss': 0.49618440866470337, 'eval_f1': 0.38045375218150085, 'eval_precision': 0.5093457943925234, 'eval_recall': 0.30362116991643456}


In [14]:
transformer_results = {}

# Note: "Real-only" can't train a real classifier (single class), so we handle it the same
# trivial way as the baseline — always predict "normal" — for a fair comparison point.
zero_preds = np.zeros(len(test_df), dtype=int)
real_only_f1 = f1_score(test_df['label'], zero_preds)
transformer_results['Real-only (no anomalies in training)'] = {
    'eval_f1': real_only_f1, 'eval_precision': 0.0, 'eval_recall': 0.0
}
print(f"Real-only (trivial baseline): F1={real_only_f1:.3f}\n")

configs = {
    'Real + Rule-based': pd.concat([real_train, rule_train], ignore_index=True),
    'Real + LLM-generated': pd.concat([real_train, llm_train], ignore_index=True),
    'Real + Both': pd.concat([real_train, rule_train, llm_train], ignore_index=True),
}

for name, train_data in configs.items():
    safe_name = name.replace(' ', '_').replace('+', 'plus')
    result = train_and_evaluate(train_data, test_df, name, f'{EVAL_DIR}/{safe_name}')
    transformer_results[name] = result
    print()

print('=== DistilBERT F1 Summary ===')
for name, res in transformer_results.items():
    print(f"{name}: F1={res['eval_f1']:.3f}, Precision={res.get('eval_precision',0):.3f}, Recall={res.get('eval_recall',0):.3f}")

Real-only (trivial baseline): F1=0.000



Map:   0%|          | 0/6264 [00:00<?, ? examples/s]

Map:   0%|          | 0/1663 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.508353,0.509771,3,0.509246,0.520349,0.498607


Real + Both final results: {'eval_loss': 0.5097705125808716, 'eval_f1': 0.5092460881934566, 'eval_precision': 0.5203488372093024, 'eval_recall': 0.4986072423398329}

=== DistilBERT F1 Summary ===
Real-only (no anomalies in training): F1=0.000, Precision=0.000, Recall=0.000
Real + Rule-based: F1=0.445, Precision=0.379, Recall=0.538
Real + LLM-generated: F1=0.380, Precision=0.509, Recall=0.304
Real + Both: F1=0.509, Precision=0.520, Recall=0.499


In [15]:
transformer_summary = pd.DataFrame([
    {'config': name, 'f1': res['eval_f1'], 'precision': res.get('eval_precision', 0), 'recall': res.get('eval_recall', 0), 'model': 'distilbert'}
    for name, res in transformer_results.items()
])
transformer_summary.to_csv(f'{EVAL_DIR}/transformer_results.csv', index=False)

final_comparison = pd.concat([
    baseline_summary.rename(columns={'f1_anomalous': 'f1'}),
    transformer_summary[['config', 'f1', 'model']]
], ignore_index=True)
final_comparison.to_csv(f'{EVAL_DIR}/final_comparison_table.csv', index=False)
print(final_comparison)

                                 config        f1         model
0  Real-only (no anomalies in training)  0.000000  tfidf_logreg
1                     Real + Rule-based  0.456674  tfidf_logreg
2                  Real + LLM-generated  0.335855  tfidf_logreg
3                           Real + Both  0.456818  tfidf_logreg
4  Real-only (no anomalies in training)  0.000000    distilbert
5                     Real + Rule-based  0.444700    distilbert
6                  Real + LLM-generated  0.380454    distilbert
7                           Real + Both  0.509246    distilbert
